##Installing Libraries

In [1]:
!pip install PyMuPDF
!pip install easyocr
!pip install python-docx
!python -m spacy download en_core_web_trf
!pip install -qqq sentence-transformers
!pip install catboost
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 30.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by se

## Importing all Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import fitz
import easyocr
import os
import docx
from PIL import Image
import io
import spacy
from spacy.matcher import PhraseMatcher
import re
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import faiss

##Loading Dataset

In [3]:
df = pd.read_csv("/content/Resume.csv")
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


##Extracting Zip File

In [4]:
with zipfile.ZipFile("/content/resumes.zip", 'r') as zip_ref:
    zip_ref.extractall("resumes")

##Performing Perfect Resume Parsing

In [5]:
# Initialize EasyOCR reader (this can be done once)
# Note: English is specified as the language. If other languages are expected, it should be adjusted.
# This might take a moment to load the model on the first run.
print("Initializing EasyOCR reader. This may take a moment...")
reader = easyocr.Reader(['en'])
print("EasyOCR reader initialized.")

# Function to extract text from PDF using PyMuPDF (fitz)
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text += page.get_text()
        doc.close()
    except Exception as e:
        print(f"Error extracting text from PDF {pdf_path} (direct text): {e}")
        text = "" # Return empty string if basic text extraction fails
    return text

# Function to extract text from DOCX using python-docx
def extract_text_from_docx(docx_path):
    text = ""
    try:
        doc = docx.Document(docx_path)
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
    except Exception as e:
        print(f"Error extracting text from DOCX {docx_path}: {e}")
        text = ""
    return text

# Function to perform OCR on a single PDF page
def ocr_pdf_page(pdf_path, page_num, reader_instance):
    text = ""
    try:
        doc = fitz.open(pdf_path)
        page = doc.load_page(page_num)
        # Render page to an image (high resolution for better OCR)
        # Using a matrix to scale up for better OCR results (e.g., 2x resolution)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img = Image.open(io.BytesIO(pix.tobytes()))

        # Perform OCR
        result = reader_instance.readtext(np.array(img))
        for (bbox, text_line, prob) in result:
            text += text_line + " "
        doc.close()
    except Exception as e:
        print(f"Error performing OCR on PDF page {page_num} of {pdf_path}: {e}")
    return text.strip() # Strip leading/trailing whitespace

# Main parsing logic
base_resumes_dir = '/content/resumes/data/data'
data_for_df = []
skipped_files = [] # To keep track of files that couldn't be parsed
total_files_processed = 0

print(f"Starting resume parsing from: {base_resumes_dir}")

# Iterate through categories (subdirectories)
for category_name in os.listdir(base_resumes_dir):
    category_path = os.path.join(base_resumes_dir, category_name)
    if os.path.isdir(category_path):
        print(f"Processing category: {category_name}")
        # Iterate through resume files in each category
        for filename in os.listdir(category_path):
            total_files_processed += 1
            file_path = os.path.join(category_path, filename)
            extracted_text = ""

            if filename.lower().endswith('.pdf'):
                # Try direct text extraction first
                extracted_text = extract_text_from_pdf(file_path)

                # If direct extraction yields little to no text, or is empty, try OCR
                if len(extracted_text.strip()) < 50: # Arbitrary threshold for "little text"
                    # print(f"Direct PDF extraction yielded little text for {filename} (length: {len(extracted_text.strip())}), trying OCR...")
                    ocr_text_pages = []
                    try:
                        doc = fitz.open(file_path)
                        for page_num in range(len(doc)):
                            page_ocr_text = ocr_pdf_page(file_path, page_num, reader)
                            if page_ocr_text:
                                ocr_text_pages.append(page_ocr_text)
                        doc.close()
                        extracted_text = " ".join(ocr_text_pages)
                        if len(extracted_text.strip()) > 50: # If OCR yields significant text
                            print(f"OCR successfully extracted text from {filename}.")
                        else:
                            print(f"OCR also yielded little text for {filename}.")
                            extracted_text = "" # If OCR is not productive, reset
                    except Exception as e:
                        print(f"Failed to process PDF for OCR (fitz.open) {file_path}: {e}")
                        extracted_text = "" # Clear extracted_text if OCR also fails

            elif filename.lower().endswith('.docx'):
                extracted_text = extract_text_from_docx(file_path)
            else:
                # print(f"Skipping unsupported file type: {filename}")
                skipped_files.append(file_path)
                continue # Skip to next file

            if extracted_text and len(extracted_text.strip()) > 0:
                data_for_df.append({
                    'Category': category_name,
                    'Filename': filename,
                    'Parsed_Resume_Text': extracted_text.strip()
                })
            else:
                # print(f"No text extracted or extracted text was empty from {filename} in category {category_name}")
                skipped_files.append(file_path)

# Create DataFrame
parsed_resumes_df = pd.DataFrame(data_for_df)

print("\n--- Parsing Summary ---")
print(f"Total files iterated: {total_files_processed}")
print(f"Total resumes successfully parsed: {len(parsed_resumes_df)}")
print(f"Total files skipped or failed to parse: {len(skipped_files)}")
print("First 5 rows of the new 'parsed_resumes_df':")
print(parsed_resumes_df.head())
print("\nDataFrame Info:")
parsed_resumes_df.info()

Initializing EasyOCR reader. This may take a moment...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR reader initialized.
Starting resume parsing from: /content/resumes/data/data
Processing category: HEALTHCARE
Processing category: CONSTRUCTION
Processing category: HR
Processing category: DIGITAL-MEDIA
Processing category: ENGINEERING
Processing category: ARTS
Processing category: DESIGNER
Processing category: APPAREL
Processing category: FINANCE
Processing category: CONSULTANT
Processing category: ACCOUNTANT
Processing category: TEACHER
Processing category: SALES
Processing category: BPO
Processing category: AGRICULTURE
Processing category: INFORMATION-TECHNOLOGY
Processing category: ADVOCATE
Processing category: CHEF
Processing category: FITNESS
Processing category: BANKING
Processing category: AVIATION
Processing category: AUTOMOBILE
Processing category: BUSINESS-DEVELOPMENT
OCR also yielded little text for 12632728.pdf.
Processing category: PUBLIC-RELATIONS

--- Parsing Summary ---
Total files ite

##Advanced Skill Extraction

In [6]:
# Layer 1: Keyword Dictionary & Layer 2: Named Entity Recognition (simulated with PhraseMatcher)
# Layer 3: Skill Ontology Mapping

# Define SKILLS_DB based on the user's prompt and expanded common skills
SKILLS_DB = [
    "python", "machine learning", "deep learning", "nlp", "sql", "docker", "aws",
    "java", "c++", "javascript", "html", "css", "excel", "word", "powerpoint", "office",
    "tableau", "power bi", "git", "github", "gitlab", "jira", "agile", "scrum",
    "data analysis", "data science", "big data", "cloud computing", "devops",
    "artificial intelligence", "r programming", "data visualization",
    "nosql databases", "sql server", "mysql", "postgresql", "mongodb", "cassandra",
    "spark", "hadoop", "kafka", "airflow", "azure", "gcp",
    "pytorch", "pandas", "tensorflow", "scikit-learn", "keras", "numpy", "scipy", "matplotlib", "seaborn",
    "natural language processing", "computer vision", "reinforcement learning",
    "statistical analysis", "modeling", "etl", "data warehousing", "api", "rest api",
    "machine learning algorithms", "deep learning frameworks", "data structures", "algorithms"
]

# Define SKILL_ONTOLOGY based on the user's prompt and expanded examples for canonicalization
SKILL_ONTOLOGY = {
    'ml': 'machine learning',
    'pytorch': 'deep learning',
    'pandas': 'python',
    'tensorflow': 'deep learning',
    'ai': 'artificial intelligence',
    'dl': 'deep learning',
    'r': 'r programming',
    'java': 'java programming',
    'c++': 'c++ programming',
    'sql server': 'sql',
    'mysql': 'sql',
    'postgresql': 'sql',
    'mongodb': 'nosql databases',
    'cassandra': 'nosql databases',
    'scikit-learn': 'machine learning',
    'nlp': 'natural language processing',
    'aws': 'amazon web services', # Mapped abbreviation to full form as per user's example
    'gcp': 'google cloud platform',
    'power bi': 'data visualization',
    'html': 'web development',
    'css': 'web development',
    'javascript': 'web development',
    'excel': 'microsoft office',
    'word': 'microsoft office',
    'powerpoint': 'microsoft office',
    'office': 'microsoft office',
    'azure': 'cloud computing',
    'spark': 'big data',
    'hadoop': 'big data',
    'kafka': 'big data',
    'airflow': 'data orchestration',
    'data scientist': 'data science',
    'data analyst': 'data analysis'
}

# Ensure nlp model is loaded for spaCy and PhraseMatcher
# The en_core_web_trf model was downloaded in a previous cell.
try:
    nlp = spacy.load("en_core_web_trf")
except Exception as e:
    print(f"Error loading spaCy model: {e}. Please ensure 'en_core_web_trf' is downloaded.")
    print("Run: !python -m spacy download en_core_web_trf")
    # In a production scenario, you might want to exit or provide a robust fallback.

# Initialize PhraseMatcher for skill extraction
matcher = PhraseMatcher(nlp.vocab)

# Create a comprehensive list of skill patterns for the matcher.
# This combines SKILLS_DB and keys/values from SKILL_ONTOLOGY for robust matching
all_skill_terms = set(SKILLS_DB)
for key, value in SKILL_ONTOLOGY.items():
    all_skill_terms.add(key)
    all_skill_terms.add(value)

# Add additional common synonyms/variations that might not be in initial lists
all_skill_terms.add("machine learning engineer")
all_skill_terms.add("deep learning engineer")
all_skill_terms.add("natural language processing engineer")
all_skill_terms.add("software development")
all_skill_terms.add("software engineer")
all_skill_terms.add("front end")
all_skill_terms.add("back end")
all_skill_terms.add("full stack")
all_skill_terms.add("computer science")
all_skill_terms.add("data engineering")

# Convert unique skill terms to spaCy Doc objects and add to matcher
patterns = [nlp.make_doc(text) for text in all_skill_terms if text] # Filter out any potential empty strings
matcher.add("SKILL", patterns)

def extract_and_map_skills(text, nlp_model, matcher_instance, skill_ontology_map):
    """
    Extracts skills from text using PhraseMatcher and maps them to canonical forms.

    Args:
        text (str): The input resume text.
        nlp_model (spacy.Language): Loaded spaCy language model.
        matcher_instance (spacy.matcher.PhraseMatcher): Initialized PhraseMatcher with skill patterns.
        skill_ontology_map (dict): Dictionary for mapping skills to canonical forms.

    Returns:
        list: A unique list of canonical skills found in the text.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp_model(text.lower()) # Process text in lowercase for case-insensitive matching
    matches = matcher_instance(doc)
    extracted_skills_set = set()

    for match_id, start, end in matches:
        span = doc[start:end] # The matched span of text
        skill = span.text

        # Apply ontology mapping (Layer 3)
        # If the skill is a key in the ontology, use its canonical value; otherwise, use the skill itself.
        mapped_skill = skill_ontology_map.get(skill, skill)

        # Add the canonicalized skill to the set
        extracted_skills_set.add(mapped_skill)

    return sorted(list(extracted_skills_set))

print("Starting skill extraction...")
# Apply the function to the 'Parsed_Resume_Text' column
parsed_resumes_df['Extracted_Skills'] = parsed_resumes_df['Parsed_Resume_Text'].apply(
    lambda x: extract_and_map_skills(x, nlp, matcher, SKILL_ONTOLOGY)
)
print("Skill extraction complete.")

# Display the first few entries with the new column
print("\nFirst 5 rows of 'parsed_resumes_df' with 'Extracted_Skills':")
print(parsed_resumes_df[['Category', 'Filename', 'Extracted_Skills']].head())

# Display distribution of skills for a sample category to verify
print("\nSample skills and their counts from 'AUTOMOBILE' category (top 10):")
sample_category_df = parsed_resumes_df[parsed_resumes_df['Category'] == 'AUTOMOBILE']
all_skills_in_category = [skill for sublist in sample_category_df['Extracted_Skills'] for skill in sublist]
skill_counts = Counter(all_skills_in_category)
print(skill_counts.most_common(10))

# Display info about the updated dataframe
print("\nDataFrame Info after skill extraction:")
parsed_resumes_df.info()

Starting skill extraction...
Skill extraction complete.

First 5 rows of 'parsed_resumes_df' with 'Extracted_Skills':
     Category      Filename                   Extracted_Skills
0  HEALTHCARE  35579812.pdf                                 []
1  HEALTHCARE  25974844.pdf                 [microsoft office]
2  HEALTHCARE  23918545.pdf  [data analysis, microsoft office]
3  HEALTHCARE  22008817.pdf  [data analysis, microsoft office]
4  HEALTHCARE  39082090.pdf                    [data analysis]

Sample skills and their counts from 'AUTOMOBILE' category (top 10):
[('microsoft office', 27), ('data analysis', 6), ('sql', 5), ('modeling', 4), ('r programming', 3), ('big data', 2), ('java programming', 2), ('python', 2), ('computer science', 1), ('data warehousing', 1)]

DataFrame Info after skill extraction:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2483 entries, 0 to 2482
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              ---------

##Semantic Understanding of Resumes

To move beyond keyword matching, we will implement **Semantic Understanding**. This involves converting the resume text and extracted skills into numerical representations (embeddings) that capture their meaning. This allows us to:

*   **Compare resumes semantically**: Find resumes that are similar in meaning, even if they use different vocabulary.
*   **Match skills more intelligently**: Understand the context of skills and identify related skills.
*   **Improve search and recommendation systems**: Provide more relevant results by understanding the underlying intent.

We will use a pre-trained `sentence-transformers` model to generate these embeddings. We will create two sets of embeddings: one for the full parsed resume text and another for the list of extracted skills.

In [7]:
# Load a pre-trained sentence transformer model for semantic understanding
# 'all-MiniLM-L6-v2' is a good balance of speed and performance
print("Loading Sentence-Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Sentence-Transformer model loaded.")

# Generate embeddings for the full parsed resume text
print("Generating embeddings for parsed resume text...")
# Ensure text is string and handle potential NaNs if any
parsed_resumes_df['Resume_Text_Embeddings'] = parsed_resumes_df['Parsed_Resume_Text'].astype(str).apply(lambda x: model.encode(x.lower()))
print("Resume text embeddings generated.")

# Generate embeddings for the extracted skills
# For skills, we join them into a single string for embedding generation per resume
print("Generating embeddings for extracted skills...")
parsed_resumes_df['Extracted_Skills_Embeddings'] = parsed_resumes_df['Extracted_Skills'].apply(lambda x: model.encode(" ".join(x).lower()))
print("Extracted skills embeddings generated.")

# Display the first few entries with the new embedding columns
print("\nFirst 5 rows of 'parsed_resumes_df' with new embedding columns:")
print(parsed_resumes_df[['Category', 'Filename', 'Resume_Text_Embeddings', 'Extracted_Skills_Embeddings']].head())

# Display info about the updated dataframe to confirm new columns
print("\nDataFrame Info after embedding generation:")
parsed_resumes_df.info()

Loading Sentence-Transformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-Transformer model loaded.
Generating embeddings for parsed resume text...
Resume text embeddings generated.
Generating embeddings for extracted skills...
Extracted skills embeddings generated.

First 5 rows of 'parsed_resumes_df' with new embedding columns:
     Category      Filename  \
0  HEALTHCARE  35579812.pdf   
1  HEALTHCARE  25974844.pdf   
2  HEALTHCARE  23918545.pdf   
3  HEALTHCARE  22008817.pdf   
4  HEALTHCARE  39082090.pdf   

                              Resume_Text_Embeddings  \
0  [0.01850917, -0.050803002, 0.04595315, 0.04138...   
1  [-0.06507887, 0.022774117, -0.024915475, -0.10...   
2  [-0.09813947, 0.03950877, 0.01598333, -0.02650...   
3  [0.0036394622, 0.049322218, -0.03905982, -0.02...   
4  [-0.015917875, 0.027318053, -0.0060300245, 0.0...   

                         Extracted_Skills_Embeddings  
0  [-0.118838415, 0.048298646, -0.0025481035, -0....  
1  [-0.0990944, 0.014773967, -0.019878885, 0.0230...  
2  [-0.022072008, 0.05770182, -0.020992167, 

##Multi-Score Engine

In [8]:
# Function to calculate cosine similarity between a query and a set of embeddings
def calculate_cosine_similarity(query_embedding, resume_embeddings):
    """
    Calculates cosine similarity between a single query embedding and multiple resume embeddings.

    Args:
        query_embedding (np.array): Embedding of the query (e.g., job description).
        resume_embeddings (list of np.array): List of embeddings for resumes.

    Returns:
        list: Cosine similarity scores for each resume.
    """
    # Reshape query_embedding to be a 2D array if it's 1D, for sklearn's cosine_similarity
    if query_embedding.ndim == 1:
        query_embedding = query_embedding.reshape(1, -1)

    # Ensure resume_embeddings are also 2D
    resume_embeddings = np.array(resume_embeddings)
    if resume_embeddings.ndim == 1:
        # This case might happen if there's only one resume, or if the embeddings themselves are scalar
        # More likely, it will be (N, embedding_dim)
        pass
    elif resume_embeddings.ndim == 0: # Handle cases where there might be no embeddings or a single scalar
        return []

    # Calculate cosine similarity. The result will be a 2D array of shape (1, N)
    similarities = cosine_similarity(query_embedding, resume_embeddings)

    # Return the first (and only) row of the similarities array
    return similarities[0].tolist()

print("Multi-Score Engine setup complete. The 'calculate_cosine_similarity' function is defined.")
print("You can now use this function to compare a job query to resume text and skill embeddings.")

Multi-Score Engine setup complete. The 'calculate_cosine_similarity' function is defined.
You can now use this function to compare a job query to resume text and skill embeddings.


In [9]:
# Define a sample job query to test the multi-score engine
job_query = "I am looking for a data scientist with strong Python, machine learning, and deep learning skills. Experience with AWS and SQL is a plus. Needs to have good communication for client-facing roles."
print(f"Job Query: {job_query}\n")

# Generate embedding for the job query
print("Generating embedding for the job query...")
job_query_embedding = model.encode(job_query.lower())
print("Job query embedding generated.")

# Calculate text similarity scores
print("Calculating text similarity scores...")
text_similarities = calculate_cosine_similarity(
    job_query_embedding,
    list(parsed_resumes_df['Resume_Text_Embeddings'])
)
parsed_resumes_df['Text_Similarity_Score'] = text_similarities
print("Text similarity scores calculated.")

# Calculate skills similarity scores
print("Calculating skills similarity scores...")
skills_similarities = calculate_cosine_similarity(
    job_query_embedding,
    list(parsed_resumes_df['Extracted_Skills_Embeddings'])
)
parsed_resumes_df['Skills_Similarity_Score'] = skills_similarities
print("Skills similarity scores calculated.")

# Combine scores (you can adjust weights based on importance)
# For example, giving more weight to skills
weight_text = 0.4
weight_skills = 0.6

parsed_resumes_df['Combined_Score'] = (
    weight_text * parsed_resumes_df['Text_Similarity_Score'] +
    weight_skills * parsed_resumes_df['Skills_Similarity_Score']
)

print("Combined scores calculated.")

# Rank resumes by the combined score
ranked_resumes = parsed_resumes_df.sort_values(by='Combined_Score', ascending=False).reset_index(drop=True)

print("\nTop 5 ranked resumes for the job query:")
print(ranked_resumes[['Category', 'Filename', 'Text_Similarity_Score', 'Skills_Similarity_Score', 'Combined_Score']].head(5))

print("\nMulti-Score Engine execution complete. The DataFrame 'parsed_resumes_df' now includes similarity scores and is ranked by 'Combined_Score'.")

Job Query: I am looking for a data scientist with strong Python, machine learning, and deep learning skills. Experience with AWS and SQL is a plus. Needs to have good communication for client-facing roles.

Generating embedding for the job query...
Job query embedding generated.
Calculating text similarity scores...
Text similarity scores calculated.
Calculating skills similarity scores...
Skills similarity scores calculated.
Combined scores calculated.

Top 5 ranked resumes for the job query:
                 Category      Filename  Text_Similarity_Score  \
0             AGRICULTURE  11813872.pdf               0.629511   
1  INFORMATION-TECHNOLOGY  83816738.pdf               0.474603   
2             AGRICULTURE  62994611.pdf               0.501094   
3              CONSULTANT  21366189.pdf               0.497323   
4  INFORMATION-TECHNOLOGY  29051656.pdf               0.433273   

   Skills_Similarity_Score  Combined_Score  
0                 0.471560        0.534741  
1             

## Experience Analysis

Quantifying professional experience is a critical component of resume screening. This section aims to extract and standardize experience-related information from the parsed resumes. While a full, highly accurate experience extraction system would be complex (involving timeline extraction, role identification, etc.), we can implement a basic version by looking for common keywords and patterns indicative of years of experience or significant roles.

For this analysis, we will focus on:

*   **Extracting potential years of experience**: Using regex to find numeric patterns followed by 'years' or 'yrs'.
*   **Identifying key experience indicators**: Looking for mentions of 'senior', 'lead', 'manager', 'principal', etc., which can hint at experience level.

This will provide another data point for resume ranking and filtering.

In [10]:
def extract_experience_indicators(text):
    if not isinstance(text, str) or not text.strip():
        return {"years_of_experience": 0, "seniority_keywords": []}

    text_lower = text.lower()

    # 1. Extract Years of Experience (basic regex)
    # Looks for patterns like '5+ years', '3 yrs', '10 year experience'
    years_match = re.search(r'(\d+)(?:\+)?\s*(?:year|yrs?|yr)\s*(?:of)?\s*(?:experience)?', text_lower)
    years_of_experience = 0
    if years_match:
        years_of_experience = int(years_match.group(1))

    # Alternative: Look for more generic number patterns near experience keywords if the above fails
    if years_of_experience == 0:
        generic_years_match = re.search(r'(\d+)\s*(?:year|yrs?|yr)', text_lower)
        if generic_years_match:
            years_of_experience = int(generic_years_match.group(1))

    # 2. Identify Seniority Keywords
    seniority_keywords = [
        'senior', 'lead', 'manager', 'principal', 'head of', 'director', 'vp',
        'architect', 'staff', 'distinguished'
    ]
    found_seniority = [keyword for keyword in seniority_keywords if keyword in text_lower]

    return {
        "years_of_experience": years_of_experience,
        "seniority_keywords": list(set(found_seniority)) # Use set to get unique keywords
    }

print("Starting experience analysis...")
# Apply the function to the 'Parsed_Resume_Text' column
experience_data = parsed_resumes_df['Parsed_Resume_Text'].apply(extract_experience_indicators)

# Expand the dictionary results into separate columns
parsed_resumes_df['Years_of_Experience'] = experience_data.apply(lambda x: x['years_of_experience'])
parsed_resumes_df['Seniority_Keywords'] = experience_data.apply(lambda x: x['seniority_keywords'])
print("Experience analysis complete.")

# Display the first few entries with the new experience columns
print("\nFirst 5 rows of 'parsed_resumes_df' with experience indicators:")
print(parsed_resumes_df[['Category', 'Filename', 'Years_of_Experience', 'Seniority_Keywords']].head())

# Display some statistics about years of experience
print("\nYears of Experience Distribution (Top 10):")
print(parsed_resumes_df['Years_of_Experience'].value_counts().head(10))

# Display info about the updated dataframe
print("\nDataFrame Info after experience analysis:")
parsed_resumes_df.info()

Starting experience analysis...
Experience analysis complete.

First 5 rows of 'parsed_resumes_df' with experience indicators:
     Category      Filename  Years_of_Experience  \
0  HEALTHCARE  35579812.pdf                   20   
1  HEALTHCARE  25974844.pdf                    0   
2  HEALTHCARE  23918545.pdf                    3   
3  HEALTHCARE  22008817.pdf                    0   
4  HEALTHCARE  39082090.pdf                   20   

                 Seniority_Keywords  
0  [manager, lead, staff, director]  
1        [manager, senior, head of]  
2                   [manager, lead]  
3  [manager, lead, staff, director]  
4            [manager, lead, staff]  

Years of Experience Distribution (Top 10):
Years_of_Experience
0     1536
10     119
5       92
20      84
15      84
3       61
4       58
2       46
7       41
8       38
Name: count, dtype: int64

DataFrame Info after experience analysis:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2483 entries, 0 to 2482
Data columns (t

##Project Relevance Engine

In [11]:
# Define keywords for identifying AI-related projects and advanced tech stack
AI_KEYWORDS = [
    "machine learning", "deep learning", "nlp", "artificial intelligence",
    "computer vision", "reinforcement learning", "neural network",
    "data science", "data scientist", "pytorch", "tensorflow", "scikit-learn",
    "ai", "ml", "dl"
]

ADVANCED_TECH_KEYWORDS = [
    "python", "java", "c++", "docker", "aws", "azure", "gcp",
    "spark", "hadoop", "kafka", "sql", "mongodb", "postgresql",
    "tableau", "power bi", "git", "api", "rest api", "kubernetes",
    "cloud computing", "devops", "web development", "backend", "frontend", "fullstack"
]

def analyze_project_relevance(resume_text, extracted_skills):
    """
    Analyzes the project relevance of a resume based on GitHub links, AI keywords,
    and advanced tech stack.

    Args:
        resume_text (str): The full parsed text of the resume.
        extracted_skills (list): A list of skills extracted from the resume.

    Returns:
        dict: A dictionary containing 'github_link_count', 'ai_keyword_count',
              'advanced_tech_score', and 'overall_project_relevance_score'.
    """
    if not isinstance(resume_text, str) or not resume_text.strip():
        return {
            "github_link_count": 0,
            "ai_keyword_count": 0,
            "advanced_tech_score": 0,
            "overall_project_relevance_score": 0
        }

    text_lower = resume_text.lower()
    skills_lower = [s.lower() for s in extracted_skills]

    # 1. GitHub Links
    # Regex to find common GitHub profile/repo links
    github_links = re.findall(r'github\.com/[\w-]{1,39}/[\w.-]{1,100}', text_lower)
    github_link_count = len(github_links)

    # 2. AI Project Keywords
    ai_keyword_count = 0
    for keyword in AI_KEYWORDS:
        ai_keyword_count += text_lower.count(keyword) # Count occurrences in text
        if keyword in skills_lower:
            ai_keyword_count += 1 # Add extra point if it's explicitly an extracted skill

    # 3. Advanced Tech Stack Score
    advanced_tech_score = 0
    for tech in ADVANCED_TECH_KEYWORDS:
        # Check if tech keyword is in extracted skills or resume text
        if tech in skills_lower or tech in text_lower:
            # Assign varying scores based on perceived 'advanced' nature
            if tech in ["python", "machine learning", "deep learning", "nlp", "data science", "cloud computing", "devops", "kubernetes"]:
                advanced_tech_score += 3
            elif tech in ["aws", "azure", "gcp", "spark", "docker", "web development"]:
                advanced_tech_score += 2
            elif tech in ["sql", "git", "api", "rest api", "backend", "frontend", "fullstack"]:
                advanced_tech_score += 1
            else:
                advanced_tech_score += 1 # Default for other advanced tech

    # Combine into an overall relevance score
    # Weights can be tuned based on desired importance
    overall_score = (
        (github_link_count * 10) +  # High value for direct project evidence
        (min(ai_keyword_count, 10) * 5) + # Cap keyword count to avoid over-weighting, then multiply
        (min(advanced_tech_score, 20) * 2) # Cap tech score to avoid over-weighting, then multiply
    )

    return {
        "github_link_count": github_link_count,
        "ai_keyword_count": ai_keyword_count,
        "advanced_tech_score": advanced_tech_score,
        "overall_project_relevance_score": overall_score
    }

print("Starting Project Relevance Analysis...")

# Apply the function to the DataFrame
# The parsed_resumes_df and Extracted_Skills are available from previous cells.
project_relevance_data = parsed_resumes_df.apply(
    lambda row: analyze_project_relevance(row['Parsed_Resume_Text'], row['Extracted_Skills']),
    axis=1
)

# Expand the dictionary results into separate columns
parsed_resumes_df['GitHub_Link_Count'] = project_relevance_data.apply(lambda x: x['github_link_count'])
parsed_resumes_df['AI_Keyword_Count'] = project_relevance_data.apply(lambda x: x['ai_keyword_count'])
parsed_resumes_df['Advanced_Tech_Score'] = project_relevance_data.apply(lambda x: x['advanced_tech_score'])
parsed_resumes_df['Overall_Project_Relevance_Score'] = project_relevance_data.apply(lambda x: x['overall_project_relevance_score'])

print("Project Relevance Analysis complete.")

# Display the first few entries with the new columns
print("\nFirst 5 rows of 'parsed_resumes_df' with project relevance indicators:")
print(parsed_resumes_df[['Category', 'Filename', 'GitHub_Link_Count', 'AI_Keyword_Count', 'Advanced_Tech_Score', 'Overall_Project_Relevance_Score']].head())

# Display some statistics about the project relevance scores
print("\nOverall Project Relevance Score Distribution (Top 10):")
print(parsed_resumes_df['Overall_Project_Relevance_Score'].value_counts().head(10))

print("\nDataFrame Info after project relevance analysis:")
parsed_resumes_df.info()

Starting Project Relevance Analysis...
Project Relevance Analysis complete.

First 5 rows of 'parsed_resumes_df' with project relevance indicators:
     Category      Filename  GitHub_Link_Count  AI_Keyword_Count  \
0  HEALTHCARE  35579812.pdf                  0                15   
1  HEALTHCARE  25974844.pdf                  0                14   
2  HEALTHCARE  23918545.pdf                  0                47   
3  HEALTHCARE  22008817.pdf                  0                17   
4  HEALTHCARE  39082090.pdf                  0                10   

   Advanced_Tech_Score  Overall_Project_Relevance_Score  
0                    0                               50  
1                    0                               50  
2                    0                               50  
3                    2                               54  
4                    0                               50  

Overall Project Relevance Score Distribution (Top 10):
Overall_Project_Relevance_Score
50    1

##ML Prediction Model

In [12]:
# --- 1. Create a placeholder for 'Education_Score' ---
# This is a simplified approach. In a real-world scenario, a more robust NLP model
# would be used to extract and score educational qualifications accurately.

def calculate_education_score(text):
    score = 0
    text_lower = text.lower()
    if "phd" in text_lower or "doctorate" in text_lower:
        score += 5
    if "master's" in text_lower or "m.s." in text_lower or "msc" in text_lower:
        score += 3
    if "bachelor's" in text_lower or "b.a." in text_lower or "b.s." in text_lower or "bachelor" in text_lower:
        score += 1
    return score

print("Calculating Education Scores...")
parsed_resumes_df['Education_Score'] = parsed_resumes_df['Parsed_Resume_Text'].apply(calculate_education_score)
print("Education Scores calculated.")

# --- 2. Create a synthetic Target Variable (for demonstration) ---
# In a real-world scenario, this would come from actual hiring decisions or expert labels.
# Here, we'll define 'High Potential' as resumes that are above a certain percentile
# for both combined similarity and project relevance.

print("Creating synthetic target variable...")
# Define thresholds based on percentiles
combined_score_threshold = parsed_resumes_df['Combined_Score'].quantile(0.75)
project_relevance_threshold = parsed_resumes_df['Overall_Project_Relevance_Score'].quantile(0.75)

parsed_resumes_df['Target_High_Potential'] = (
    (parsed_resumes_df['Combined_Score'] >= combined_score_threshold) &
    (parsed_resumes_df['Overall_Project_Relevance_Score'] >= project_relevance_threshold)
).astype(int)

print(f"Target variable 'Target_High_Potential' created. Class distribution:\n{parsed_resumes_df['Target_High_Potential'].value_counts()}")

# --- 3. Define Features (X) and Target (y) ---
# Using the features suggested by the user and the newly created ones.

features = [
    'Skills_Similarity_Score',
    'Text_Similarity_Score', # semantic_score
    'Years_of_Experience',
    'Education_Score',
    'Overall_Project_Relevance_Score' # project_score
]

X = parsed_resumes_df[features]
y = parsed_resumes_df['Target_High_Potential']

print(f"Features selected: {features}")

# --- 4. Split data into training and testing sets ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Data split into training ({len(X_train)} samples) and testing ({len(X_test)} samples) sets.")

# --- 5. Train and Evaluate Models ---
models = {
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear'),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0, iterations=100)
}

results = {}

print("\nStarting model training and evaluation...")
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        'accuracy': accuracy,
        'classification_report': report,
        'roc_auc': roc_auc
    }

    print(f"{name} - Accuracy: {accuracy:.4f}")
    print(f"{name} - ROC AUC: {roc_auc:.4f}")
    # print(f"Classification Report for {name}:\n{classification_report(y_test, y_pred)}")

print("\n--- Model Training and Evaluation Complete ---")
print("\nSummary of Model Performance:")
for name, res in results.items():
    print(f"  {name}:")
    print(f"    Accuracy: {res['accuracy']:.4f}")
    print(f"    ROC AUC: {res['roc_auc']:.4f}")
    print(f"    F1-score (Class 1): {res['classification_report']['1']['f1-score']:.4f}")

print("\nDataFrame Info after adding Education_Score and Target_High_Potential:")
parsed_resumes_df.info()

Calculating Education Scores...
Education Scores calculated.
Creating synthetic target variable...
Target variable 'Target_High_Potential' created. Class distribution:
Target_High_Potential
0    2131
1     352
Name: count, dtype: int64
Features selected: ['Skills_Similarity_Score', 'Text_Similarity_Score', 'Years_of_Experience', 'Education_Score', 'Overall_Project_Relevance_Score']
Data split into training (1986 samples) and testing (497 samples) sets.

Starting model training and evaluation...

Training Logistic Regression...
Logistic Regression - Accuracy: 0.9095
Logistic Regression - ROC AUC: 0.9878

Training Random Forest...
Random Forest - Accuracy: 0.9960
Random Forest - ROC AUC: 0.9997

Training XGBoost...
XGBoost - Accuracy: 0.9940
XGBoost - ROC AUC: 0.9999

Training LightGBM...
[LightGBM] [Info] Number of positive: 282, number of negative: 1704


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:39:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000359 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 441
[LightGBM] [Info] Number of data points in the train set: 1986, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.141994 -> initscore=-1.798827
[LightGBM] [Info] Start training from score -1.798827
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

## Top Candidates Ranking

In [13]:
ranked_resumes_display = ranked_resumes.copy()
ranked_resumes_display['Score'] = (ranked_resumes_display['Combined_Score'] * 100).round(2).astype(str) + '%'
ranked_resumes_display = ranked_resumes_display.rename(columns={'Filename': 'Candidate'})
display(ranked_resumes_display[['Candidate', 'Score']].head(10))

,Candidate,Score
0,11813872.pdf,53.47%
1,83816738.pdf,45.37%
2,62994611.pdf,45.15%
3,21366189.pdf,43.97%
4,29051656.pdf,43.9%
5,20674668.pdf,43.37%
6,20824105.pdf,43.05%
7,93349646.pdf,42.71%
8,23464505.pdf,42.2%
9,12011623.pdf,41.49%


## FAISS Vector Database for Semantic Search

To enable instant and scalable semantic search for resumes, we will utilize FAISS (Facebook AI Similarity Search). FAISS is a library for efficient similarity search and clustering of dense vectors. It allows us to quickly find the most similar resumes to a given query by searching through their vector embeddings.

### Preparing Embeddings for FAISS

We have generated separate embeddings for the full resume text and the extracted skills. For FAISS, we will combine these into a single, comprehensive embedding vector for each resume. This can be done by concatenating the two embedding vectors.

In [14]:
# Ensure embeddings are in a consistent format (e.g., numpy arrays)
# First, check the dimensions of the existing embeddings
embedding_dim_text = parsed_resumes_df['Resume_Text_Embeddings'].iloc[0].shape[0]
embedding_dim_skills = parsed_resumes_df['Extracted_Skills_Embeddings'].iloc[0].shape[0]

print(f"Resume Text Embedding Dimension: {embedding_dim_text}")
print(f"Extracted Skills Embedding Dimension: {embedding_dim_skills}")

# Combine text and skills embeddings for each resume
# We will concatenate them to capture both semantic aspects.
# Ensure all embeddings are numpy arrays for concatenation.

combined_embeddings_list = []
for index, row in parsed_resumes_df.iterrows():
    text_emb = row['Resume_Text_Embeddings']
    skills_emb = row['Extracted_Skills_Embeddings']

    # Ensure they are numpy arrays
    if not isinstance(text_emb, np.ndarray):
        text_emb = np.array(text_emb)
    if not isinstance(skills_emb, np.ndarray):
        skills_emb = np.array(skills_emb)

    combined_emb = np.concatenate((text_emb, skills_emb))
    combined_embeddings_list.append(combined_emb)

# Convert the list of combined embeddings to a 2D numpy array
# FAISS requires input vectors to be float32
combined_resume_embeddings = np.array(combined_embeddings_list).astype('float32')

final_embedding_dim = combined_resume_embeddings.shape[1]
print(f"Combined Resume Embedding Dimension for FAISS: {final_embedding_dim}")
print(f"Shape of combined_resume_embeddings: {combined_resume_embeddings.shape}")

Resume Text Embedding Dimension: 384
Extracted Skills Embedding Dimension: 384
Combined Resume Embedding Dimension for FAISS: 768
Shape of combined_resume_embeddings: (2483, 768)


### Building the FAISS Index

Now, we will build a FAISS index using the combined embeddings. We'll use a simple `IndexFlatL2` index, which performs an exhaustive search using L2 (Euclidean) distance, suitable for semantic similarity. For larger datasets, more advanced indices like `IndexIVFFlat` could be used for speedup.

In [15]:
# Build a FAISS index
# Using IndexFlatL2 for exhaustive search (Euclidean distance)
index = faiss.IndexFlatL2(final_embedding_dim)

# Add the combined resume embeddings to the index
index.add(combined_resume_embeddings)

print(f"FAISS index built successfully. Total vectors in index: {index.ntotal}")

# Save the FAISS index to disk (optional, but good practice)
faiss.write_index(index, "resume_faiss_index.bin")
print("FAISS index saved to 'resume_faiss_index.bin'")

FAISS index built successfully. Total vectors in index: 2483
FAISS index saved to 'resume_faiss_index.bin'


### Performing Semantic Search with FAISS

Now we can use the FAISS index to find the most similar resumes to a given job query. We'll encode the job query using the same `SentenceTransformer` model, combine its text and skill embeddings (if applicable), and then query the FAISS index.

In [17]:
# Re-initialize the SentenceTransformer model to ensure the correct model is used
# This is necessary because the 'model' variable was overwritten by the ML training loop in a previous cell.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to perform semantic search using the FAISS index
def semantic_search_faiss(query_text, num_results=5, model_instance=model, faiss_index=index, df_source=parsed_resumes_df):
    # 1. Generate embeddings for the query text
    query_embedding_text = model_instance.encode(query_text.lower())

    # For simplicity, let's assume we can also derive 'skills' from the query itself or just use text embedding for query
    # A more advanced approach would involve parsing skills from the job query too.
    # For now, we'll assume job_query_embedding as our combined query vector

    # To match the combined_resume_embeddings structure, we need to create a combined query embedding
    # We'll use the same SentenceTransformer model for both text and a simplified skill representation from the query.
    # In a real scenario, you might have specific skills from the job description.
    # Here, let's just re-encode a simplified 'skill' aspect from the query text for demonstration consistency.

    # For demonstration, let's assume the job query implicitly contains skill information that can be embedded.
    # We will re-use the job_query_embedding for both text and 'mock' skill part for combination,
    # as we don't have separate skill extraction for the query.

    query_embedding_skills_mock = model_instance.encode(query_text.lower()) # Re-use for consistency of dimension
    combined_query_embedding = np.concatenate((query_embedding_text, query_embedding_skills_mock)).astype('float32')

    # Reshape for FAISS query (1, dimension)
    combined_query_embedding = combined_query_embedding.reshape(1, -1)

    # 2. Perform search
    distances, indices = faiss_index.search(combined_query_embedding, num_results) # D: distances, I: indices

    # 3. Retrieve results from the original DataFrame
    results = []
    for i, idx in enumerate(indices[0]):
        resume_info = df_source.iloc[idx].copy()
        resume_info['FAISS_Distance'] = distances[0][i] # L2 distance
        results.append(resume_info)

    return pd.DataFrame(results)

# Example Usage:
job_query_for_search = "Experienced Python Developer with expertise in machine learning and cloud platforms like AWS."
print(f"Searching for: {job_query_for_search}")

faiss_search_results = semantic_search_faiss(job_query_for_search, num_results=10)

print("\nTop 10 resumes found via FAISS semantic search:")
# Display relevant columns from the search results
display(faiss_search_results[['Filename', 'Category', 'FAISS_Distance', 'Combined_Score']].sort_values(by='FAISS_Distance').reset_index(drop=True))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Searching for: Experienced Python Developer with expertise in machine learning and cloud platforms like AWS.

Top 10 resumes found via FAISS semantic search:


,Filename,Category,FAISS_Distance,Combined_Score
0,11813872.pdf,AGRICULTURE,1.886430,0.534741
1,62994611.pdf,AGRICULTURE,1.939386,0.451459
2,20824105.pdf,INFORMATION-TECHNOLOGY,1.972587,0.430543
3,83816738.pdf,INFORMATION-TECHNOLOGY,2.057434,0.453691
4,20674668.pdf,INFORMATION-TECHNOLOGY,2.144840,0.433715
5,26071861.pdf,ADVOCATE,2.165735,0.333795
6,26480367.pdf,INFORMATION-TECHNOLOGY,2.194655,0.348054
7,12144825.pdf,AVIATION,2.198870,0.363340
8,21156767.pdf,CONSULTANT,2.243658,0.408551
9,13405733.pdf,INFORMATION-TECHNOLOGY,2.245017,0.398058


## Explainable AI: Matched and Missing Skills

To provide more transparency and interpretability to the ranking system, we will incorporate an Explainable AI component. This component will identify and display the specific skills from a job query that are **matched** in a candidate's resume and those that are **missing**. This helps recruiters understand the rationale behind a candidate's fit beyond just a numerical score.

### Skill Comparison Function

We'll define a utility function to compare a set of required job skills with the skills extracted from a resume, identifying which skills are present and which are absent.

In [18]:
def compare_skills(job_skills, resume_skills):
    """
    Compares job skills with resume skills and identifies matched and missing skills.

    Args:
        job_skills (list): A list of canonical skills required for the job.
        resume_skills (list): A list of canonical skills extracted from the resume.

    Returns:
        tuple: (matched_skills, missing_skills)
    """
    job_skills_set = set(job_skills)
    resume_skills_set = set(resume_skills)

    matched_skills = sorted(list(job_skills_set.intersection(resume_skills_set)))
    missing_skills = sorted(list(job_skills_set.difference(resume_skills_set)))

    return matched_skills, missing_skills

print("Skill comparison function 'compare_skills' defined.")

Skill comparison function 'compare_skills' defined.


### Updating Semantic Search with Skill Explanation

Now, we'll enhance our `semantic_search_faiss` function to first extract skills from the job query and then use the `compare_skills` function to add 'Matched Skills' and 'Missing Skills' columns to the results.

In [19]:
# Re-defining the semantic_search_faiss function to include skill comparison
def semantic_search_faiss_with_explanation(query_text, num_results=5, model_instance=model, faiss_index=index, df_source=parsed_resumes_df, nlp_model=nlp, matcher_instance=matcher, skill_ontology_map=SKILL_ONTOLOGY):
    # 1. Generate embeddings for the query text
    query_embedding_text = model_instance.encode(query_text.lower())
    query_embedding_skills_mock = model_instance.encode(query_text.lower())
    combined_query_embedding = np.concatenate((query_embedding_text, query_embedding_skills_mock)).astype('float32')
    combined_query_embedding = combined_query_embedding.reshape(1, -1)

    # 2. Extract skills from the job query
    job_query_skills = extract_and_map_skills(query_text, nlp_model, matcher_instance, skill_ontology_map)

    # 3. Perform FAISS search
    distances, indices = faiss_index.search(combined_query_embedding, num_results)

    # 4. Retrieve results and add skill comparison
    results = []
    for i, idx in enumerate(indices[0]):
        resume_info = df_source.iloc[idx].copy()
        resume_info['FAISS_Distance'] = distances[0][i]

        # Perform skill comparison
        resume_extracted_skills = resume_info['Extracted_Skills']
        matched, missing = compare_skills(job_query_skills, resume_extracted_skills)
        resume_info['Matched_Skills'] = matched
        resume_info['Missing_Skills'] = missing

        results.append(resume_info)

    return pd.DataFrame(results)

# Example Usage with the updated function:
job_query_for_search_explained = "Experienced Python Developer with expertise in machine learning and cloud platforms like AWS. Familiarity with Kubernetes and Azure is a plus."
print(f"Searching for: {job_query_for_search_explained}")

faiss_search_results_explained = semantic_search_faiss_with_explanation(job_query_for_search_explained, num_results=5)

print("\nTop 5 resumes found via FAISS semantic search with skill explanations:")
# Display relevant columns from the search results, including new skill explanation columns
display(faiss_search_results_explained[['Filename', 'Category', 'FAISS_Distance', 'Matched_Skills', 'Missing_Skills']].sort_values(by='FAISS_Distance').reset_index(drop=True))

Searching for: Experienced Python Developer with expertise in machine learning and cloud platforms like AWS. Familiarity with Kubernetes and Azure is a plus.

Top 5 resumes found via FAISS semantic search with skill explanations:


,Filename,Category,FAISS_Distance,Matched_Skills,Missing_Skills
0,62994611.pdf,AGRICULTURE,2.093088,"[machine learning, python]","[amazon web services, cloud computing]"
1,20824105.pdf,INFORMATION-TECHNOLOGY,2.117883,"[amazon web services, python]","[cloud computing, machine learning]"
2,11813872.pdf,AGRICULTURE,2.168103,"[amazon web services, python]","[cloud computing, machine learning]"
3,83816738.pdf,INFORMATION-TECHNOLOGY,2.250566,[amazon web services],"[cloud computing, machine learning, python]"
4,26071861.pdf,ADVOCATE,2.342961,[python],"[amazon web services, cloud computing, machine..."


### Making Predictions with the Best Model

Now, let's use the best performing model (CatBoost) to make predictions on our entire dataset and compare them with the actual 'High Potential' labels we created.

In [20]:
# Select the best performing model (e.g., CatBoost) from the trained models
best_model = models["CatBoost"]

# Make predictions on the full feature set X
parsed_resumes_df['Predicted_High_Potential'] = best_model.predict(X)

# Display a comparison of Actual vs. Predicted for a sample of resumes
print("Actual vs. Predicted High Potential for a sample of resumes:")
display(parsed_resumes_df[['Filename', 'Category', 'Target_High_Potential', 'Predicted_High_Potential']].sample(10, random_state=42))

# Optionally, display evaluation metrics on the full dataset
print("\nEvaluation Metrics on the full dataset (using the best model):")
full_accuracy = accuracy_score(y, parsed_resumes_df['Predicted_High_Potential'])
full_report = classification_report(y, parsed_resumes_df['Predicted_High_Potential'], output_dict=True)
full_roc_auc = roc_auc_score(y, best_model.predict_proba(X)[:, 1])

print(f"Accuracy on full dataset: {full_accuracy:.4f}")
print(f"ROC AUC on full dataset: {full_roc_auc:.4f}")
print(f"F1-score (Class 1) on full dataset: {full_report['1']['f1-score']:.4f}")

Actual vs. Predicted High Potential for a sample of resumes:


,Filename,Category,Target_High_Potential,Predicted_High_Potential
1544,10641230.pdf,INFORMATION-TECHNOLOGY,1,1
1308,70892619.pdf,TEACHER,0,0
1091,23513618.pdf,ACCOUNTANT,0,0
1359,16223371.pdf,SALES,0,0
2265,18293620.pdf,BUSINESS-DEVELOPMENT,0,0
990,22351830.pdf,CONSULTANT,1,1
173,12693146.pdf,CONSTRUCTION,0,0
480,62071407.pdf,ENGINEERING,0,0
1476,12341902.pdf,AGRICULTURE,0,0
188,29894080.pdf,CONSTRUCTION,0,0



Evaluation Metrics on the full dataset (using the best model):
Accuracy on full dataset: 0.9988
ROC AUC on full dataset: 1.0000
F1-score (Class 1) on full dataset: 0.9957


## Saving Important Files for Deployment

To deploy our resume screening system, we need to save the trained FAISS index and the best-performing machine learning model (CatBoost) so they can be loaded and used in a production environment without retraining.

In [21]:
# Save the trained CatBoost model
best_model.save_model("catboost_model.cbm")
print("CatBoost model saved to 'catboost_model.cbm'")

# The FAISS index was already saved in a previous step to 'resume_faiss_index.bin'
# We can confirm its presence or re-save it if needed.
# faiss.write_index(index, "resume_faiss_index.bin")
# print("FAISS index re-confirmed/saved to 'resume_faiss_index.bin'")

CatBoost model saved to 'catboost_model.cbm'
